# 03. Pipeline de Categorização

Este notebook executa o tratamento e categorização dos pulsares com base em suas propriedades físicas, lendo os dados brutos e exportando um arquivo limpo e rotulado para a pasta `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# configurando caminhos
BASE_DIR = Path("../../").resolve()
RAW_DIR = BASE_DIR / "data" / "raw" / "atnf_raw_arquivos"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# cria a pasta processed se não existir
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# carrega o dado bruto mais recente
arquivo_raw = sorted(RAW_DIR.glob("atnf_catalog_raw_*.csv"))[-1]
print(f"Lendo arquivo bruto: {arquivo_raw.name}")
df = pd.read_csv(arquivo_raw)

Lendo arquivo bruto: atnf_catalog_raw_2026-08-21.csv


## Lógica de Categorização
A classificação respeita a física e propriedades observadas no ATNF:

In [ ]:
def classificar_tipo(row):
    tipo = row["TYPE"]
    periodo = row["P0"]
    
    # magnetares (AXP ou SGR)
    if pd.notna(tipo) and "AXP" in str(tipo):
        return "magnetar"
    
    # tipos com tag explícita no catálogo
    if pd.notna(tipo) and "XINS" in str(tipo):
        return "XINS"
    if pd.notna(tipo) and "RRAT" in str(tipo):
        return "RRAT"
    if pd.notna(tipo) and "NRAD" in str(tipo):
        return "NRAD"
    if pd.notna(tipo) and "HE" in str(tipo):
        return "HE"
    
    # MSP (Millisecond Pulsar): derivado do período físico (P < 30ms)
    if pd.notna(periodo) and periodo < 0.03:
        return "MSP"
    
    # todos os demais: pulsares normais do disco
    return "normal"

df["tipo_principal"] = df.apply(classificar_tipo, axis=1)

print("Distribuição final dos tipos físicos:")
print(df["tipo_principal"].value_counts())

Distribuição final dos tipos físicos:
tipo_principal
normal      3100
MSP          608
HE           250
RRAT         220
NRAD          93
magnetar      24
XINS           8
Name: count, dtype: int64


## Salvar Arquivo Processado

In [6]:
saida = PROCESSED_DIR / "catalog_categorized.csv"
df.to_csv(saida, index=False)
print(f" Arquivo processado e salvo com sucesso em:\n  {saida}")

 Arquivo processado e salvo com sucesso em:
  /home/kiwii/Documentos/estrelas de neutrons/neutron-star-observational-database/data/processed/catalog_categorized.csv
